### KIKO_Real-Time Detection and Filtering of Abusive Content

Feature extraction


In [109]:
import pandas as pd
import numpy as np

In [110]:
df1 = pd.read_csv('D:\\ML CODES\\IBM\\Train_Cleaned\\Airline_train_cleaned.csv')
df2 = pd.read_csv('D:\\ML CODES\\IBM\\Train_Cleaned\\text.tweeet_cleaned.csv')
# df3 = pd.read_csv('D:\\ML CODES\\IBM\\Train_Cleaned\\train_cleaned.csv')
df4 = pd.read_csv('D:\\ML CODES\\IBM\\Train_Cleaned\\train_tweet_cleaned.csv')

#Concatenating the dataframes

df = pd.concat([df1, df2, df4], ignore_index=True)

In [111]:
df.head()

,sentiment,text
0,neutral,What said
1,positive,plus youve added commercials to the experienc...
2,neutral,I didnt today Must mean I need to take anothe...
3,negative,its really aggressive to blast obnoxious ente...
4,negative,and its a really big bad thing about it


In [112]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45463 entries, 0 to 45462
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   sentiment  45463 non-null  object
 1   text       45462 non-null  object
dtypes: object(2)
memory usage: 710.5+ KB


In [113]:
print(df['sentiment'].value_counts())

sentiment
negative    17869
neutral     15612
positive    11982
Name: count, dtype: int64


In [114]:
df.duplicated().sum()

285

In [122]:
df.drop_duplicates(inplace=True)

In [120]:
df.dropna(inplace=True)

In [121]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

In [123]:
# Normalize labels and keep only valid rows
df = df.dropna(subset=['text', 'sentiment']).copy()
df['sentiment'] = df['sentiment'].astype(str).str.strip().str.lower()

x = df['text'].astype(str)
y = df['sentiment']

x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,
    random_state=42,
    stratify=y
  )

In [ ]:
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LinearSVC())
])

grid = {
    "tfidf__sublinear_tf": [True],
    "tfidf__strip_accents": ["unicode"],
    "tfidf__analyzer": ["word", "char_wb"],
    "tfidf__ngram_range": [(1, 2), (3, 5)],
    "tfidf__max_df": [0.85, 0.95],
    "tfidf__min_df": [1, 2],
    "tfidf__stop_words": [None, "english"],
    "clf__C": [0.1, 0.5, 1, 2],
    "clf__class_weight": [None, "balanced"],
    "clf__loss": ["squared_hinge"]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=grid,
    cv=cv,
    n_jobs=-1,
    verbose=2,
    scoring="accuracy"
  )

grid_search.fit(x_train, y_train)
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best CV Accuracy: {grid_search.best_score_:.4f}")

best_model = grid_search.best_estimator_
y_pred = best_model.predict(x_test)

print(f"Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred))

Fitting 5 folds for each of 256 candidates, totalling 1280 fits
